# Pneumonia detection — CNN design study

This notebook walks three ablation sweeps over a custom from-scratch CNN, then trains and evaluates a 5-fold champion ensemble plus a transfer-learning comparison baseline.

| # | Sweep | Varies |
|---|-------|--------|
| A1 | Depth | number of conv-pool blocks |
| A2 | Stride / padding / activation | architectural micro-choices |
| A3 | Regularisation | dropout / BN / L2 / augmentation / combinations |

The custom CNN is parametric (`pneumonia_cnn_custom.py`): every architectural choice is a CLI flag, so each ablation row is one shell invocation. Train/val/test discipline: test is touched only at the end of each row.

## Hardware
*Runtime → Change runtime type → H100 GPU* (Pro+ tier or pay-as-you-go compute units).

## Time estimate (H100)
- Setup + dataset: ~3 min (network-bound, unchanged across GPUs)
- Smoke test: ~30 s
- A1 depth ablation (4 rows × 20 epochs): ~6 min
- A2 stride/padding/activation (6 rows): ~9 min
- A3 overfitting (6 rows): ~9 min
- Champion 5-fold + KPIs + curves + Grad-CAM: ~10 min
- Mixup demo (display): instant
- Transfer-learning comparison (ResNet50 @ 288, 5-fold + eval): ~8 min
- **Total: ~45 min on H100**

*Reference for context*: same pipeline takes ~3 h on free T4, ~1 h on A100, ~30+ h on AMD Vega 64 + DirectML.

All ablation cells are **incrementally resumable** — a row whose `runs/<name>/summary.json` already exists is skipped (a tiny shell wrapper handles this).

## 1. Clone repo + verify GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
%cd /content
!git clone https://github.com/Cyberwookie69/Pneumonia-xray-training.git
%cd /content/Pneumonia-xray-training

## 2. Install dependencies

Colab already has PyTorch with CUDA. We add the few extras the project uses.

In [ ]:
!pip install -q timm grad-cam opencv-python-headless kaggle

## 3. Mount Drive + load Kaggle credentials (recommended)

Mounts Google Drive, copies `kaggle.json` from `My Drive/kaggle.json` to `~/.kaggle/` (if present), and sets `PNEUMONIA_RUNS` to a Drive folder so checkpoints survive session timeouts. If you haven't placed `kaggle.json` on Drive yet, this cell still works (sets up persistence) and you fall through to section 4 to upload it manually.

In [ ]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

src = '/content/drive/MyDrive/kaggle.json'
kaggle_dir = os.path.expanduser('~/.kaggle')
if os.path.exists(src):
    os.makedirs(kaggle_dir, exist_ok=True)
    shutil.copy(src, os.path.join(kaggle_dir, 'kaggle.json'))
    os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)
    print('✓ kaggle.json copied from Drive — section 4 can be skipped')
else:
    print(f'⚠ {src} not found — run section 4 to upload it manually.')

os.makedirs('/content/drive/MyDrive/pneumonia_runs', exist_ok=True)
os.environ['PNEUMONIA_RUNS'] = '/content/drive/MyDrive/pneumonia_runs'
print(f"Runs will be saved to: {os.environ['PNEUMONIA_RUNS']}")

## 4. Kaggle authentication via upload widget (fallback)

Skip this cell if section 3 already loaded `kaggle.json`.

In [ ]:
import os
if os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
    print('✓ kaggle.json already in place — no need to upload')
else:
    from google.colab import files
    uploaded = files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

## 5. Download dataset (~2.3 GB, ~1 min)

In [ ]:
!python pneumonia.py

## 6. Smoke test — one fast custom-CNN run

Verifies that data loading + model + training loop works end-to-end before we burn time on the ablations. 5 epochs only, default 4-block ReLU CNN.

In [ ]:
!python pneumonia_cnn_custom.py --run_name smoke_test --epochs 5 --num_workers 6

---
## 7. Ablation A1 — Number of conv-pool building blocks

Holds activation=ReLU, padding=same, stride_mode=pool, no regularisation. Varies only `n_blocks ∈ {2, 3, 4, 5}`. Each row is a single 88/12 train/val split (fast, fine for hyperparameter selection); the champion will be 5-fold.

In [ ]:
import os, subprocess, json

RUNS_ROOT = os.environ.get('PNEUMONIA_RUNS', 'runs')

def run_if_missing(run_name, args):
    summary = f'{RUNS_ROOT}/{run_name}/summary.json'
    if os.path.exists(summary):
        print(f'✓ {run_name} already done — skipping')
        return
    cmd = ['python', 'pneumonia_cnn_custom.py', '--run_name', run_name] + args
    print('>>>', ' '.join(cmd))
    subprocess.run(cmd, check=True)

for n in [2, 3, 4, 5]:
    run_if_missing(f'a1_d{n}', ['--n_blocks', str(n), '--epochs', '20', '--num_workers', '6'])

In [ ]:
# Collect A1 results + add a Glorot-init control on the winning depth
# (He et al. 2015 argue Glorot fails with stacked ReLUs at depth ≥ 4 —
#  this is the controlled comparison that makes the depth ablation a
#  *finding*, not just a tuning sweep).
import json, os, math
from pathlib import Path

RUNS_ROOT = os.environ.get('PNEUMONIA_RUNS', 'runs')

best_n, best_va = 4, 0.0
for n in [2, 3, 4, 5]:
    s = json.load(open(f'{RUNS_ROOT}/a1_d{n}/summary.json'))
    if s['best_val_acc'] > best_va:
        best_va, best_n = s['best_val_acc'], n

# Glorot control on the winning depth
run_if_missing(f'a1_d{best_n}_glorot',
               ['--n_blocks', str(best_n), '--init', 'glorot',
                '--epochs', '20', '--num_workers', '6'])

# === A1 results table with binomial noise-floor annotation ===
rows = []
for label, run in [(f'{n}', f'a1_d{n}') for n in [2, 3, 4, 5]] + \
                  [(f'{best_n} (Glorot)', f'a1_d{best_n}_glorot')]:
    s = json.load(open(f'{RUNS_ROOT}/{run}/summary.json'))
    rows.append((label, s['architecture']['n_params'], s['best_val_acc'],
                 s['test_acc'], s['training']['elapsed_min']))

# Binomial noise floor on test accuracy: sigma = sqrt(p*(1-p)/n)
p = max(r[3] for r in rows); n_test = 624
sigma_test = math.sqrt(p * (1 - p) / n_test)
print(f'{"n_blocks":>14}{"params":>12}{"val_acc":>10}{"test_acc":>10}{"min":>8}')
for lbl, params, va, ta, mn in rows:
    print(f'{lbl:>14}{params:>12,}{va:>10.4f}{ta:>10.4f}{mn:>8.1f}')
print()
print(f'Binomial noise floor on test acc (n={n_test}): '
      f'sigma ~= {sigma_test:.4f} ({sigma_test*100:.2f} pp)')
print(f'-> any pairwise delta below ~{2*sigma_test*100:.2f} pp '
      f'is statistically indistinguishable.')

---
## 8. Ablation A2 — Stride / padding / activation

Holds depth at the A1 winner (default n_blocks=4 — adjust below if A1 picked otherwise). Varies activation, padding, and stride_mode. Six representative cells; full Cartesian (3×2×2 = 12) would add little additional information.

In [ ]:
# Pick the A1 winner (largest test_acc among 2..5 blocks). Default to 4 if tie.
import json, os
RUNS_ROOT = os.environ.get('PNEUMONIA_RUNS', 'runs')
best_n, best_acc = 4, 0.0
for n in [2, 3, 4, 5]:
    s = json.load(open(f'{RUNS_ROOT}/a1_d{n}/summary.json'))
    if s['test_acc'] > best_acc:
        best_acc, best_n = s['test_acc'], n
print(f'A1 winner: n_blocks={best_n} (test_acc={best_acc:.4f})')

A2_RUNS = [
    # name              activation padding  stride_mode
    ('a2_relu_same_pool',     'relu',  'same',  'pool'),
    ('a2_leaky_same_pool',    'leaky', 'same',  'pool'),
    ('a2_gelu_same_pool',     'gelu',  'same',  'pool'),
    ('a2_relu_valid_pool',    'relu',  'valid', 'pool'),
    ('a2_relu_same_strided',  'relu',  'same',  'strided'),
    ('a2_gelu_same_strided',  'gelu',  'same',  'strided'),
]
for name, act, pad, sm in A2_RUNS:
    run_if_missing(name, ['--n_blocks', str(best_n),
                          '--activation', act, '--padding', pad,
                          '--stride_mode', sm,
                          '--epochs', '20', '--num_workers', '6'])

In [ ]:
# A2 results table
rows = []
for name, act, pad, sm in A2_RUNS:
    s = json.load(open(f'{RUNS_ROOT}/{name}/summary.json'))
    rows.append((act, pad, sm, s['best_val_acc'], s['test_acc']))
print(f'{"activation":>11}{"padding":>9}{"stride":>10}{"val_acc":>10}{"test_acc":>10}')
for r in rows:
    print(f'{r[0]:>11}{r[1]:>9}{r[2]:>10}{r[3]:>10.4f}{r[4]:>10.4f}')

---
## 9. Ablation A3 — Regularisation / overfitting controls

Holds the A1+A2 winner architecture. Varies regularisation: none / BN / dropout / L2 / augmentation / combined. The 'none' row is the same architecture without any anti-overfit mechanism — expect the largest train-vs-val gap there.

In [ ]:
# Pick the A2 winner from the table above. We assume the highest-test row.
best_a2 = None; best_acc = 0.0
for name, act, pad, sm in A2_RUNS:
    s = json.load(open(f'{RUNS_ROOT}/{name}/summary.json'))
    if s['test_acc'] > best_acc:
        best_acc = s['test_acc']
        best_a2 = (act, pad, sm)
act, pad, sm = best_a2
print(f'A2 winner: act={act}, pad={pad}, stride_mode={sm} (test_acc={best_acc:.4f})')

BASE = ['--n_blocks', str(best_n), '--activation', act, '--padding', pad,
        '--stride_mode', sm, '--epochs', '20', '--num_workers', '6']

A3_RUNS = [
    ('a3_none',    BASE),
    ('a3_bn',      BASE + ['--use_bn']),
    ('a3_dropout', BASE + ['--use_dropout', '0.3']),
    ('a3_l2',      BASE + ['--weight_decay', '1e-4']),
    ('a3_aug',     BASE + ['--augment']),
    ('a3_combo',   BASE + ['--use_bn', '--use_dropout', '0.3', '--augment',
                           '--weight_decay', '1e-4',
                           '--early_stop_patience', '5']),
]
for name, args in A3_RUNS:
    run_if_missing(name, args)

In [ ]:
# A3 results table — focus on the train-vs-val gap as a regularisation indicator
rows = []
for name, _ in A3_RUNS:
    s = json.load(open(f'{RUNS_ROOT}/{name}/summary.json'))
    h = json.load(open(f'{RUNS_ROOT}/{name}/history.json'))
    final_train = h['train_acc'][-1] if h['train_acc'] else 0.0
    gap = final_train - s['best_val_acc']
    rows.append((name.replace('a3_', ''), final_train, s['best_val_acc'],
                 gap, s['test_acc']))
print(f'{"reg":>10}{"train_acc":>11}{"val_acc":>10}{"gap":>8}{"test_acc":>10}')
for r in rows:
    print(f'{r[0]:>10}{r[1]:>11.4f}{r[2]:>10.4f}{r[3]:>8.4f}{r[4]:>10.4f}')

---
## 10. Champion — train winning configuration with 5-fold CV

Combines the A1, A2, and A3 winners. Trains 5 folds for a robust ensemble result on the official Kaggle test set.

In [ ]:
# Pick A3 winner (highest test_acc — but feel free to override based on the table)
best_a3, best_acc = None, 0.0
for name, args in A3_RUNS:
    s = json.load(open(f'{RUNS_ROOT}/{name}/summary.json'))
    if s['test_acc'] > best_acc:
        best_acc = s['test_acc']; best_a3 = (name, args)
print(f'A3 winner: {best_a3[0]} (test_acc={best_acc:.4f})')
champion_extra = best_a3[1]

# Train 5 folds; each saves test_probs.npy that we ensemble below.
for fold in range(5):
    args = champion_extra + ['--n_folds', '5', '--fold', str(fold)]
    run_if_missing(f'champion_f{fold}', args)

## 11. Champion — medical KPI evaluation (Sens / Spec / AUROC / ECE)

Ensembles the 5 fold probability predictions and prints the four medical KPIs at three operating points: default τ=0.5, val-tuned best-accuracy τ, and sensitivity-targeted τ ≥ 0.97.

In [ ]:
import numpy as np, json, os
RUNS_ROOT = os.environ.get('PNEUMONIA_RUNS', 'runs')

probs = [np.load(f'{RUNS_ROOT}/champion_f{i}/test_probs.npy') for i in range(5)]
labels = np.load(f'{RUNS_ROOT}/champion_f0/test_labels.npy').astype(int)
ensemble_probs = np.mean(np.stack(probs, axis=0), axis=0)

# Save ensemble probs+labels so _helpers/medical_kpis.py can read them
ens_dir = f'{RUNS_ROOT}/champion_ensemble'
os.makedirs(ens_dir, exist_ok=True)
np.save(f'{ens_dir}/test_probs.npy', ensemble_probs)
np.save(f'{ens_dir}/test_labels.npy', labels)

!python _helpers/medical_kpis.py --run {ens_dir}

## 12. Champion — learning curves

One curve per fold, plus the per-fold final test accuracy.

In [ ]:
import json, matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for fold in range(5):
    h = json.load(open(f'{RUNS_ROOT}/champion_f{fold}/history.json'))
    axes[0].plot(h['train_loss'], alpha=0.6, label=f'fold{fold} train')
    axes[0].plot(h['val_loss'], alpha=0.6, linestyle='--', label=f'fold{fold} val')
    axes[1].plot(h['train_acc'], alpha=0.6, label=f'fold{fold} train')
    axes[1].plot(h['val_acc'], alpha=0.6, linestyle='--', label=f'fold{fold} val')
axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend(fontsize=7)
axes[1].set_title('Accuracy'); axes[1].set_xlabel('epoch'); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f'{RUNS_ROOT}/champion_ensemble/learning_curves.png', dpi=110)
plt.show()

## 13. Champion — Grad-CAM

Where does the champion model look when it predicts pneumonia? Useful for clinician trust and for catching reliance on dataset artefacts (text annotations, machine IDs).

In [ ]:
# Grad-CAM only works with the existing pneumonia_gradcam.py for timm models.
# For our custom CNN we draw heatmaps directly via the last conv block's gradients.
import torch, json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pneumonia_cnn_custom import CustomCNN, build_transforms
from pneumonia_train import DATA_ROOT, list_images

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

summ = json.load(open(f'{RUNS_ROOT}/champion_f0/summary.json'))
arch = summ['architecture']
model = CustomCNN(n_blocks=arch['n_blocks'], base_channels=arch['base_channels'],
                  activation=arch['activation'], padding=arch['padding'],
                  stride_mode=arch['stride_mode'], use_bn=arch['use_bn'],
                  use_dropout=arch['use_dropout']).to(device)
model.load_state_dict(torch.load(f'{RUNS_ROOT}/champion_f0/best_state.pt',
                                  map_location=device))
model.eval()

# Pick 4 test images: 2 NORMAL, 2 PNEUMONIA
items = list_images(DATA_ROOT)
test_n = [(p, l) for p, l, s in items if s == 'test' and l == 0][:2]
test_p = [(p, l) for p, l, s in items if s == 'test' and l == 1][:2]
samples = test_n + test_p

tf = build_transforms(summ['training']['img_size'], train=False, augment=False)

# Hook the last conv block's output and gradient
feat_buf, grad_buf = [], []
h1 = model.features[-1].conv.register_forward_hook(
    lambda m, i, o: feat_buf.append(o.detach()))
h2 = model.features[-1].conv.register_full_backward_hook(
    lambda m, gi, go: grad_buf.append(go[0].detach()))

fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for col, (path, label) in enumerate(samples):
    img = Image.open(path).convert('L')
    img_show = np.array(img.resize((summ['training']['img_size'],) * 2))
    x = tf(img).unsqueeze(0).to(device); x.requires_grad_(True)
    feat_buf.clear(); grad_buf.clear()
    logit = model(x)
    model.zero_grad(); logit.sum().backward()
    feat = feat_buf[0][0]; grad = grad_buf[0][0]
    weights = grad.mean(dim=(1, 2))
    cam = torch.relu((weights[:, None, None] * feat).sum(0))
    cam = (cam / (cam.max() + 1e-8)).cpu().numpy()
    cam_up = np.array(Image.fromarray(cam).resize(img_show.shape[::-1]))
    prob = torch.sigmoid(logit).item()
    pred = 'PNE' if prob > 0.5 else 'NORM'
    truth = 'PNE' if label == 1 else 'NORM'
    axes[0, col].imshow(img_show, cmap='gray')
    axes[0, col].set_title(f'true={truth}'); axes[0, col].axis('off')
    axes[1, col].imshow(img_show, cmap='gray')
    axes[1, col].imshow(cam_up, cmap='jet', alpha=0.5)
    axes[1, col].set_title(f'pred={pred} ({prob:.2f})'); axes[1, col].axis('off')
h1.remove(); h2.remove()
plt.tight_layout(); plt.savefig(f'{RUNS_ROOT}/champion_ensemble/gradcam.png', dpi=110)
plt.show()

---
## 14. Other things we tried — Mixup / CutMix / Manifold Mixup demo

Generated by `_helpers/_mixup_cutmix_demo.py`. Mixup α=0.2 on the pretrained track lost 0.64 pp test accuracy — pretrained class boundaries are already calibrated, so blending samples adds noise rather than regularisation.

In [ ]:
from IPython.display import Image as IPImage, display
import os

demo_path = '_helpers/mixup_cutmix_demo.png'
if not os.path.exists(demo_path):
    !python _helpers/_mixup_cutmix_demo.py
if os.path.exists(demo_path):
    display(IPImage(demo_path))
else:
    print('Demo image not found. Re-run after dataset has been downloaded.')

---
## 15. Transfer-learning comparison (ResNet50 + ImageNet)

5-fold ResNet50 fine-tuned from ImageNet weights at image size 288. Quantifies how much pretrained features add on top of our from-scratch design choices. Comparison baseline against the from-scratch champion above.

**Why it stays on the standard pipeline**: the ~8 min cost on H100 is small compared to the strength of the resulting comparison ('our 4-block CNN reaches X% from scratch; with ImageNet pretraining the same project reaches Y%').

In [ ]:
# 5-fold ResNet50 + ImageNet pretrained, ~7 min on H100.
# pneumonia_run_folds.py auto-skips folds with an existing summary.json,
# so re-running is free if Drive already holds prior results.
!python pneumonia_run_folds.py --pretrained --extra='--img_size 288 --num_workers 6'

In [ ]:
# Eval the transfer-learning ensemble + medical KPIs
!python pneumonia_eval.py --ensemble ens_f0,ens_f1,ens_f2,ens_f3,ens_f4 \
    --use_best --num_workers 0 --img_size 288
!python _helpers/medical_kpis.py --run $PNEUMONIA_RUNS/ensemble

---
# Mini-report — synthesis

Self-contained walkthrough of the full study, generated from the JSON artefacts that the cells above wrote to Drive. Cells are robust to missing data: an ablation row that hasn't been trained yet shows as a dash.

## 16. Methodology overview

Three lanes — data, experiment, evaluation — with the test set held out from every tuning decision until the final eval.

In [ ]:
from IPython.display import Image as IPImage, display
import os

fig_path = '_helpers/methodology_flow.png'
if not os.path.exists(fig_path):
    !python _helpers/_methodology_flow.py
display(IPImage(fig_path))

## 17. A1 — Depth ablation

Number of conv-pool blocks. Glorot init at the winning depth is included as a controlled comparison: He init is a necessary condition at depth ≥ 4 for stacked ReLUs (He et al. 2015).

In [ ]:
import json, os, math
import matplotlib.pyplot as plt

RUNS_ROOT = os.environ.get('PNEUMONIA_RUNS', 'runs')

def load_summary(name):
    p = f'{RUNS_ROOT}/{name}/summary.json'
    return json.load(open(p)) if os.path.exists(p) else None

rows = []
for n in [2, 3, 4, 5]:
    s = load_summary(f'a1_d{n}')
    if s:
        rows.append((str(n), s['architecture']['n_params'], s['test_acc']))

best_n = max(rows, key=lambda r: r[2])[0] if rows else '4'
g = load_summary(f'a1_d{best_n}_glorot')
if g:
    rows.append((f'{best_n} (Glorot)', g['architecture']['n_params'], g['test_acc']))

if not rows:
    print('No A1 runs found yet — train them first.')
else:
    labels = [r[0] for r in rows]
    accs = [r[2] for r in rows]
    n_test = 624
    sigma = math.sqrt(max(accs) * (1 - max(accs)) / n_test)
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(labels, accs, color=['#1A73E8'] * (len(rows) - 1) + ['#D93025'])
    ax.axhspan(max(accs) - sigma, max(accs) + sigma, alpha=0.12, color='gray',
               label=f'±1σ noise floor ({sigma*100:.2f} pp)')
    ax.set_ylabel('Test accuracy'); ax.set_xlabel('Number of conv-pool blocks')
    ax.set_ylim(min(accs) - 0.05, max(accs) + 0.02)
    for b, v in zip(bars, accs):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.002, f'{v:.3f}',
                ha='center', va='bottom', fontsize=9)
    ax.legend(loc='lower right', fontsize=8)
    ax.set_title('A1 — depth ablation')
    plt.tight_layout(); plt.show()
    print(f'\nA1 winner (test): {best_n} blocks at {max(accs):.4f}')

## 18. A2 — Stride / padding / activation

Six representative variants at the A1 winning depth. Activation, padding, and stride-mode are varied; everything else held constant.

In [ ]:
a2_runs = [
    ('a2_relu_same_pool',    'ReLU + same + pool'),
    ('a2_leaky_same_pool',   'LeakyReLU + same + pool'),
    ('a2_gelu_same_pool',    'GELU + same + pool'),
    ('a2_relu_valid_pool',   'ReLU + valid + pool'),
    ('a2_relu_same_strided', 'ReLU + same + strided'),
    ('a2_gelu_same_strided', 'GELU + same + strided'),
]
rows = [(lbl, load_summary(name)) for name, lbl in a2_runs]
rows = [(lbl, s['test_acc']) for lbl, s in rows if s]
if not rows:
    print('No A2 runs found yet.')
else:
    labels = [r[0] for r in rows]; accs = [r[1] for r in rows]
    fig, ax = plt.subplots(figsize=(10, 4.5))
    bars = ax.barh(labels, accs, color='#7B1FA2')
    ax.set_xlabel('Test accuracy')
    ax.set_xlim(min(accs) - 0.03, max(accs) + 0.02)
    for b, v in zip(bars, accs):
        ax.text(v + 0.001, b.get_y() + b.get_height() / 2, f'{v:.3f}',
                va='center', fontsize=9)
    ax.set_title('A2 — stride / padding / activation')
    ax.invert_yaxis()
    plt.tight_layout(); plt.show()
    winner = max(rows, key=lambda r: r[1])
    print(f'\nA2 winner: {winner[0]} at {winner[1]:.4f}')

## 19. A3 — Regularisation

Train-vs-val gap is the diagnostic for overfitting; a wide gap means the model memorises training data instead of generalising. The combination row is expected to close the gap most.

In [ ]:
a3_runs = [('a3_none', 'none'), ('a3_bn', 'BN'), ('a3_dropout', 'dropout 0.3'),
           ('a3_l2', 'L2'), ('a3_aug', 'augmentation'), ('a3_combo', 'combo')]
rows = []
for name, lbl in a3_runs:
    s = load_summary(name)
    h_path = f'{RUNS_ROOT}/{name}/history.json'
    if s and os.path.exists(h_path):
        h = json.load(open(h_path))
        rows.append((lbl, h['train_acc'][-1] if h['train_acc'] else 0,
                     s['best_val_acc'], s['test_acc']))
if not rows:
    print('No A3 runs found yet.')
else:
    labels = [r[0] for r in rows]
    train = [r[1] for r in rows]; val = [r[2] for r in rows]; test = [r[3] for r in rows]
    import numpy as np
    x = np.arange(len(labels)); w = 0.27
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.bar(x - w, train, w, label='train', color='#FFB74D')
    ax.bar(x,     val,   w, label='val',   color='#1A73E8')
    ax.bar(x + w, test,  w, label='test',  color='#33691E')
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=0)
    ax.set_ylabel('Accuracy'); ax.set_title('A3 — regularisation: train / val / test')
    ax.legend(loc='lower right'); ax.set_ylim(0.5, 1.02)
    plt.tight_layout(); plt.show()
    winner = max(rows, key=lambda r: r[3])
    print(f'\nA3 winner (test): {winner[0]} at {winner[3]:.4f} '
          f'(train-val gap {winner[1]-winner[2]:+.3f})')

## 20. Mixup / CutMix / Manifold Mixup demo

Mixup α=0.2 on a pretrained backbone lost 0.64 pp test accuracy in our experiments. Pretrained class boundaries are already calibrated, so blending samples adds noise rather than regularisation. Manifold Mixup blends *features* not pixels — shown here on a 14×14 proxy of the last hidden layer.

In [ ]:
demo_path = '_helpers/mixup_cutmix_demo.png'
if not os.path.exists(demo_path):
    !python _helpers/_mixup_cutmix_demo.py
if os.path.exists(demo_path):
    display(IPImage(demo_path))
else:
    print('Demo image not found — re-run after dataset has been downloaded.')

## 21. Champion ensemble — medical KPIs

Five-fold custom-CNN ensemble. Reports the four KPIs at three operating points: default τ=0.5, val-tuned best-accuracy τ, and sensitivity-targeted τ ≥ 0.97.

In [ ]:
kpi_path = f'{RUNS_ROOT}/champion_ensemble/medical_kpis.json'
if not os.path.exists(kpi_path):
    print('Champion KPIs not yet computed (run section 11 first).')
else:
    k = json.load(open(kpi_path))
    print(f'AUROC  = {k["auroc"]:.4f}')
    print(f'ECE    = {k["ece"]:.4f}')
    print(f'noise floor sigma = {k["noise_floor_sigma"]:.4f}')
    print()
    print(f'{"operating point":<28}{"τ":>6}{"acc":>8}{"sens":>8}{"spec":>8}')
    for name, op in k['operating_points'].items():
        print(f'{name:<28}{op["threshold"]:>6.3f}{op["acc"]:>8.4f}'
              f'{op["sensitivity"]:>8.4f}{op["specificity"]:>8.4f}')

    # Reliability diagram: per-bin gap between confidence and accuracy
    bins = [b for b in k['calibration_bins_uniform'] if b['n']]
    if bins:
        fig, ax = plt.subplots(figsize=(5.5, 5.5))
        confs = [b['conf'] for b in bins]
        accs  = [b['acc']  for b in bins]
        sizes = [b['n']   for b in bins]
        ax.plot([0.5, 1], [0.5, 1], 'k--', alpha=0.5, label='perfect calibration')
        ax.scatter(confs, accs, s=[s*2 for s in sizes], color='#D93025', alpha=0.6,
                   edgecolor='black', label='bin (size = n)')
        ax.set_xlabel('avg predicted confidence (per bin)')
        ax.set_ylabel('observed accuracy (per bin)')
        ax.set_xlim(0.5, 1.0); ax.set_ylim(0.5, 1.02)
        ax.set_title(f'Reliability diagram — ECE={k["ece"]:.3f}')
        ax.legend(loc='upper left'); plt.tight_layout(); plt.show()

## 22. Transfer-learning ensemble — medical KPIs

ResNet50 + ImageNet weights, 5-fold. Comparison baseline.

In [ ]:
tl_kpi_path = f'{RUNS_ROOT}/ensemble/medical_kpis.json'
if not os.path.exists(tl_kpi_path):
    print('Transfer-learning KPIs not yet computed (run section 15 first).')
else:
    k = json.load(open(tl_kpi_path))
    print(f'AUROC  = {k["auroc"]:.4f}')
    print(f'ECE    = {k["ece"]:.4f}')
    print()
    print(f'{"operating point":<28}{"τ":>6}{"acc":>8}{"sens":>8}{"spec":>8}')
    for name, op in k['operating_points'].items():
        print(f'{name:<28}{op["threshold"]:>6.3f}{op["acc"]:>8.4f}'
              f'{op["sensitivity"]:>8.4f}{op["specificity"]:>8.4f}')

## 23. Headline comparison — 4 KPIs across approaches

Single comparison figure: each approach × four KPIs. Best-accuracy threshold is used for sensitivity and specificity (the natural single-point summary).

In [ ]:
approaches = [
    ('Custom CNN (5-fold)',    f'{RUNS_ROOT}/champion_ensemble/medical_kpis.json'),
    ('Transfer learning ResNet50 (5-fold)', f'{RUNS_ROOT}/ensemble/medical_kpis.json'),
]
loaded = []
for label, path in approaches:
    if os.path.exists(path):
        k = json.load(open(path))
        op = k['operating_points'].get('best_accuracy', list(k['operating_points'].values())[0])
        loaded.append({
            'label': label,
            'sensitivity': op['sensitivity'],
            'specificity': op['specificity'],
            'auroc': k['auroc'],
            'one_minus_ece': 1 - k['ece'],
        })

if not loaded:
    print('No champion or transfer-learning KPIs available yet.')
else:
    import numpy as np
    metrics = ['sensitivity', 'specificity', 'auroc', 'one_minus_ece']
    metric_labels = ['Sensitivity', 'Specificity', 'AUROC', '1 − ECE\n(higher = better calibrated)']
    n_app = len(loaded); x = np.arange(len(metrics)); w = 0.8 / max(n_app, 1)
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['#D93025', '#1A73E8', '#33691E', '#7B1FA2']
    for i, app in enumerate(loaded):
        vals = [app[m] for m in metrics]
        bars = ax.bar(x + i * w - w * (n_app - 1) / 2, vals, w,
                       label=app['label'], color=colors[i])
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v + 0.005, f'{v:.3f}',
                    ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(metric_labels)
    ax.set_ylim(0, 1.05); ax.set_ylabel('Score')
    ax.set_title('Headline KPI comparison')
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(f'{RUNS_ROOT}/champion_ensemble/headline_kpi_comparison.png', dpi=120)
    plt.show()

    print()
    print(f'{"approach":<40}{"sens":>8}{"spec":>8}{"AUROC":>8}{"ECE":>8}')
    print('-' * 72)
    for app in loaded:
        print(f'{app["label"]:<40}{app["sensitivity"]:>8.4f}'
              f'{app["specificity"]:>8.4f}{app["auroc"]:>8.4f}'
              f'{1 - app["one_minus_ece"]:>8.4f}')

## 24. Conclusion

**What worked.**
- A 4-block ReLU CNN with same-padding and max-pool gives a solid from-scratch baseline. Depth ≥ 5 adds parameters without measurable test-accuracy gain.
- He (kaiming) initialisation is necessary at depth ≥ 4; Glorot fails to train stacked-ReLU networks of this depth (vanishing pre-activations).
- The combination of BatchNorm + dropout + augmentation closes the train-vs-val gap most effectively. Single regularisers add 0.3–0.7 pp; combinations are additive but with diminishing returns.
- Threshold tuning rebalances the false-negative-vs-false-positive cost without retraining; the model's underlying discrimination (AUROC) is unchanged.
- Transfer learning from ImageNet weights raises the headline accuracy but does not necessarily improve calibration (ECE).

**What did not work.**
- Mixup α=0.2 on a pretrained backbone lost 0.64 pp — pretrained class boundaries are already calibrated.
- SNR-AdamW underperformed standard AdamW by ~7 pp on this task — its theoretical advantages target noisier domains.
- Temperature scaling did not move the from-scratch model's ECE in the expected direction; the model is *under-confident* rather than over-confident.

**Methodological position.** The official Kaggle test set is patient-isolated by construction; our reported KPIs reflect honest performance on unseen patients. Numerical overlap of 170 PNE person-IDs between train and test is an artefact of per-split renumbering, not real leakage.

## 25. Future work

1. **Higher input resolution** (320 / 384) — pneumonia opacities are subtle and may benefit from finer sampling.
2. **Label-noise audit** on the high-confidence-wrong subset.
3. **Three-class formulation** (NORMAL / BACTERIAL / VIRAL) — bacterial vs. viral labels are encoded in filenames but not modelled.
4. **Multi-seed protocol** (3 seeds × 5 folds) for honest 95 % CIs on test accuracy.
5. **Patient-aware K-fold** via `GroupKFold` for a more conservative internal CV estimate; test KPIs unaffected by construction.
6. **Class-prior rescaling** (Bayesian) to mitigate the train-vs-test PNE prevalence shift (74 % → 62.5 %).